# 고객 신용평가 데이터 EDA (A / B / C / EXT 그룹)

전처리가 끝난 4개의 파일을 각각 독립적으로 탐색적 데이터 분석(EDA)합니다.

| 그룹 | 파일명 | 내용 |
|---|---|---|
| A | `model_A_financial.csv` | 금융(소득/대출/신용조회/신용국 이력 관련) 변수 |
| B | `model_B_nonfinancial.csv` | 비금융(연령/직업/소득유형/조직유형) 변수 |
| C | `model_C_nonfinancial_new.csv` | 비금융(자산/자녀/사회적 관계/학력/주거) 변수 |
| EXT | `model_EXT_source.csv` | 외부 신용평가 점수(EXT_SOURCE_1~3) |

모든 파일은 `SK_ID_CURR`(고객 ID), `TARGET`(연체 여부)을 공통 키로 갖지만, 개인정보 보호를 위해 컬럼에 `_A`, `_B` 같은 접미사를 붙여 하나의 wide 테이블로 합치지 않고 **파일 단위로 분리 저장**되어 있습니다. 이 노트북에서도 그룹을 하나로 합치지 않고, 그룹별로 독립적으로 EDA를 수행합니다.

## 1. 환경 설정 & 라이브러리

In [ ]:
!pip install -q pandas numpy matplotlib seaborn scipy

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 150)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.unicode_minus'] = False

### 한글 폰트 설정 (Colab 환경)
그래프 제목/라벨에 한글이 깨지지 않도록 나눔고딕 폰트를 설치합니다.

In [ ]:
!apt-get -qq install fonts-nanum > /dev/null 2>&1
import matplotlib.font_manager as fm

font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fm.fontManager.addfont(font_path)
plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

## 2. 데이터 업로드

아래 두 가지 방법 중 하나를 사용하세요.

**방법 A. 구글 드라이브 마운트 (권장, 파일이 클 때 편함)**
드라이브에 4개 CSV를 올려둔 뒤, 아래 `DATA_DIR` 경로를 실제 폴더 경로로 수정하세요.

**방법 B. 직접 업로드**
`files.upload()` 셀의 주석을 해제해서 로컬에서 바로 4개 파일을 업로드할 수도 있습니다.

In [ ]:
# 방법 A: 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import glob

TARGET_FILENAMES = [
    'model_A_financial.csv',
    'model_B_nonfinancial.csv',
    'model_C_nonfinancial_new.csv',
    'model_EXT_source.csv',
]

# DATA_DIR을 몰라도 되도록 마운트된 드라이브 전체에서 파일명을 재귀 검색합니다.
# 파일이 많으면 시간이 걸릴 수 있으니, 위치를 알고 있다면 DATA_DIR을 직접 지정해서
# os.path.join(DATA_DIR, fname) 방식으로 바꿔도 됩니다.
print('드라이브에서 파일 검색 중... (파일이 많으면 다소 걸립니다)')
found = {}
for fname in TARGET_FILENAMES:
    matches = glob.glob(f'/content/drive/MyDrive/**/{fname}', recursive=True)
    if matches:
        found[fname] = matches[0]
        if len(matches) > 1:
            print(f'[안내] {fname} 이(가) {len(matches)}곳에서 발견되어 첫 번째 경로를 사용합니다: {matches}')
    else:
        print(f'[경고] {fname} 을(를) 드라이브에서 찾지 못했습니다.')

missing = [f for f in TARGET_FILENAMES if f not in found]
if missing:
    raise FileNotFoundError(
        f'다음 파일을 MyDrive에서 찾지 못했습니다: {missing}\n'
        '해당 CSV들을 구글 드라이브에 업로드했는지 확인하거나, '
        'DATA_PATHS를 직접 지정하거나, 아래 방법 B(직접 업로드) 셀을 사용하세요.'
    )

DATA_PATHS = {
    'A_financial':     found['model_A_financial.csv'],
    'B_nonfinancial':  found['model_B_nonfinancial.csv'],
    'C_nonfinancial':  found['model_C_nonfinancial_new.csv'],
    'EXT_source':      found['model_EXT_source.csv'],
}
for name, path in DATA_PATHS.items():
    print(f'{name:16s} -> {path}')

In [ ]:
# 방법 B: 직접 업로드 (방법 A 대신 이 셀을 사용할 경우 주석 해제)
# from google.colab import files
# uploaded = files.upload()  # model_A_financial.csv, model_B_nonfinancial.csv,
#                             # model_C_nonfinancial_new.csv, model_EXT_source.csv 4개를 선택
# DATA_PATHS = {
#     'A_financial':    'model_A_financial.csv',
#     'B_nonfinancial': 'model_B_nonfinancial.csv',
#     'C_nonfinancial': 'model_C_nonfinancial_new.csv',
#     'EXT_source':     'model_EXT_source.csv',
# }

In [ ]:
dfs = {}
for name, path in DATA_PATHS.items():
    dfs[name] = pd.read_csv(path)
    print(f'{name:16s} shape={dfs[name].shape}  path={path}')

## 3. 그룹 간 키 정합성 체크

컬럼을 하나로 합치지는 않지만, `SK_ID_CURR`과 `TARGET`만 이용해 네 파일이 동일한 고객·동일한 라벨을 가리키는지만 가볍게 확인합니다 (다른 개인정보 컬럼은 전혀 사용하지 않으므로 분리 저장 취지를 해치지 않습니다).

In [ ]:
KEY_COL, TARGET_COL = 'SK_ID_CURR', 'TARGET'

key_frames = {name: df[[KEY_COL, TARGET_COL]].drop_duplicates() for name, df in dfs.items()}
base_name = list(key_frames.keys())[0]
base = key_frames[base_name]

print(f'기준 그룹: {base_name} (n={len(base)})')
for name, kdf in key_frames.items():
    if name == base_name:
        continue
    merged = base.merge(kdf, on=KEY_COL, suffixes=('', '_chk'), how='inner')
    mismatch = (merged[TARGET_COL] != merged[f'{TARGET_COL}_chk']).sum()
    print(f'{base_name} vs {name}: 공통 SK_ID_CURR={len(merged)}, TARGET 불일치={mismatch}, '
          f'{name}에만 있는 ID={len(kdf) - len(merged)}, {base_name}에만 있는 ID={len(base) - len(merged)}')

## 4. 그룹별 EDA 함수 정의

각 그룹의 컬럼을 자동으로 분류해서 처리합니다.
- **연속형**: 수치형이면서 고유값이 많은 컬럼 (예: 소득, 신용점수)
- **범주형**: object 타입이거나 고유값이 15개 이하인 컬럼
- **이진 플래그**: {0,1} 또는 True/False만 갖는 컬럼 (원-핫 인코딩 결과 다수 포함)

In [ ]:
OUTPUT_DIR = '/content/eda_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)


def classify_columns(df, id_col=KEY_COL, target_col=TARGET_COL, low_card_threshold=15):
    cols = [c for c in df.columns if c not in (id_col, target_col)]
    binary_cols, categorical_cols, continuous_cols = [], [], []
    for c in cols:
        s = df[c]
        if s.dtype == bool:
            binary_cols.append(c)
            continue
        nunique = s.nunique(dropna=True)
        uniq_vals = set(s.dropna().unique().tolist())
        if nunique <= 2 and uniq_vals.issubset({0, 1, 0.0, 1.0}):
            binary_cols.append(c)
        elif s.dtype == object or nunique <= low_card_threshold:
            categorical_cols.append(c)
        else:
            continuous_cols.append(c)
    return binary_cols, categorical_cols, continuous_cols


def missing_summary(df):
    miss = df.isnull().sum()
    pct = 100 * miss / len(df)
    out = pd.DataFrame({'n_missing': miss, 'pct_missing': pct})
    return out[out['n_missing'] > 0].sort_values('pct_missing', ascending=False)


def iqr_outlier_summary(df, cols):
    rows = []
    for c in cols:
        s = df[c].dropna()
        if len(s) == 0:
            continue
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n_out = int(((s < lower) | (s > upper)).sum())
        rows.append({'column': c, 'n_outliers': n_out,
                     'pct_outliers': round(100 * n_out / len(s), 2),
                     'skew': round(s.skew(), 3)})
    return pd.DataFrame(rows).sort_values('pct_outliers', ascending=False)

In [ ]:
def run_eda(name, df, id_col=KEY_COL, target_col=TARGET_COL, save=True):
    print('=' * 90)
    print(f'[{name}] EDA 시작 - shape: {df.shape}')
    print('=' * 90)

    out_dir = os.path.join(OUTPUT_DIR, name)
    if save:
        os.makedirs(out_dir, exist_ok=True)

    # 1) 기본 정보
    print('\n[1] dtype 구성')
    print(df.dtypes.value_counts())
    display(df.head(3))

    # 2) 결측치
    print('\n[2] 결측치 요약')
    miss = missing_summary(df)
    if len(miss):
        display(miss)
        if save:
            miss.to_csv(f'{out_dir}/missing_summary.csv')
        plt.figure(figsize=(9, max(2.5, 0.3 * len(miss))))
        sns.barplot(x=miss['pct_missing'], y=miss.index, color='steelblue')
        plt.xlabel('결측 비율 (%)')
        plt.title(f'[{name}] 결측치 비율')
        plt.tight_layout()
        if save:
            plt.savefig(f'{out_dir}/missing_bar.png', dpi=120)
        plt.show()
    else:
        print('결측치 없음')

    # 3) 중복
    print('\n[3] 중복 확인')
    n_dup_rows = int(df.duplicated().sum())
    n_dup_id = int(df[id_col].duplicated().sum()) if id_col in df.columns else None
    print(f'완전 중복 행: {n_dup_rows}건 / 중복 SK_ID_CURR: {n_dup_id}건')

    # 4) TARGET 분포
    if target_col in df.columns:
        print('\n[4] TARGET 분포')
        target_dist = df[target_col].value_counts(normalize=True).mul(100).round(2)
        print(target_dist)
        plt.figure(figsize=(4.5, 4))
        sns.countplot(x=df[target_col])
        plt.title(f"[{name}] TARGET 분포 (연체율 {target_dist.get(1, 0):.2f}%)")
        plt.tight_layout()
        if save:
            plt.savefig(f'{out_dir}/target_dist.png', dpi=120)
        plt.show()

    # 5) 컬럼 분류
    binary_cols, categorical_cols, continuous_cols = classify_columns(df, id_col, target_col)
    print(f'\n[5] 컬럼 분류 -> 연속형 {len(continuous_cols)} / '
          f'범주형 {len(categorical_cols)} / 이진 플래그 {len(binary_cols)}')

    # 6) 연속형 변수
    if continuous_cols:
        print('\n[6] 연속형 변수 기술통계')
        desc = df[continuous_cols].describe().T
        display(desc)
        if save:
            desc.to_csv(f'{out_dir}/continuous_describe.csv')

        ncols = 4
        nrows = int(np.ceil(len(continuous_cols) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
        axes = np.array(axes).reshape(-1)
        for i, c in enumerate(continuous_cols):
            sns.histplot(df[c].dropna(), bins=40, kde=True, ax=axes[i], color='teal')
            axes[i].set_title(c, fontsize=9)
        for j in range(len(continuous_cols), len(axes)):
            fig.delaxes(axes[j])
        plt.suptitle(f'[{name}] 연속형 변수 분포', y=1.02)
        plt.tight_layout()
        if save:
            plt.savefig(f'{out_dir}/continuous_hist.png', dpi=120, bbox_inches='tight')
        plt.show()

        print('\n[6-1] IQR 기준 이상치 요약')
        outlier_df = iqr_outlier_summary(df, continuous_cols)
        display(outlier_df)
        if save:
            outlier_df.to_csv(f'{out_dir}/outlier_summary.csv', index=False)

        corr_cols = continuous_cols + ([target_col] if target_col in df.columns else [])
        corr = df[corr_cols].corr()
        plt.figure(figsize=(max(6, 0.5 * len(corr_cols)), max(5, 0.5 * len(corr_cols))))
        sns.heatmap(corr, cmap='coolwarm', center=0, annot=len(corr_cols) <= 20, fmt='.2f')
        plt.title(f'[{name}] 연속형 변수 상관관계')
        plt.tight_layout()
        if save:
            plt.savefig(f'{out_dir}/correlation_heatmap.png', dpi=120)
        plt.show()

        if target_col in corr.columns:
            target_corr = corr[target_col].drop(target_col)
            target_corr = target_corr.reindex(target_corr.abs().sort_values(ascending=False).index)
            print('\n[6-2] TARGET과 상관관계 높은 변수 Top 10')
            display(target_corr.head(10))

    # 7) 범주형 변수
    if categorical_cols:
        print('\n[7] 범주형 변수 분포')
        ncols = 3
        nrows = int(np.ceil(len(categorical_cols) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows))
        axes = np.array(axes).reshape(-1)
        for i, c in enumerate(categorical_cols):
            order = df[c].value_counts().index
            sns.countplot(x=df[c], order=order, ax=axes[i], color='slateblue')
            axes[i].set_title(c, fontsize=9)
            axes[i].tick_params(axis='x', rotation=45)
        for j in range(len(categorical_cols), len(axes)):
            fig.delaxes(axes[j])
        plt.suptitle(f'[{name}] 범주형 변수 분포', y=1.02)
        plt.tight_layout()
        if save:
            plt.savefig(f'{out_dir}/categorical_dist.png', dpi=120, bbox_inches='tight')
        plt.show()

        if target_col in df.columns:
            print('\n[7-1] 범주별 TARGET 비율(%)')
            for c in categorical_cols:
                rate = df.groupby(c)[target_col].mean().mul(100).sort_values(ascending=False)
                print(f'- {c}')
                print(rate.round(2).to_string())
                print()

    # 8) 이진 플래그 변수
    if binary_cols:
        print(f'\n[8] 이진 플래그 변수 {len(binary_cols)}개 요약')
        bin_df = df[binary_cols].apply(lambda s: s.astype(int) if s.dtype == bool else s)
        prevalence = bin_df.mean().mul(100).sort_values(ascending=False)
        prevalence.name = 'pct_flag_1'
        display(prevalence)
        if save:
            prevalence.to_frame().to_csv(f'{out_dir}/binary_flag_prevalence.csv')

        plt.figure(figsize=(8, max(3, 0.25 * len(binary_cols))))
        sns.barplot(x=prevalence.values, y=prevalence.index, color='darkorange')
        plt.xlabel('플래그=1 비율 (%)')
        plt.title(f'[{name}] 이진 플래그 변수 비율')
        plt.tight_layout()
        if save:
            plt.savefig(f'{out_dir}/binary_flag_prevalence.png', dpi=120)
        plt.show()

        if target_col in df.columns:
            rows = {}
            for c in binary_cols:
                s = bin_df[c]
                rate1 = df.loc[s == 1, target_col].mean() * 100
                rate0 = df.loc[s == 0, target_col].mean() * 100
                rows[c] = rate1 - rate0
            diff = pd.Series(rows)
            diff = diff.reindex(diff.abs().sort_values(ascending=False).index)
            print('\n[8-1] 플래그=1 vs 0 TARGET 비율 차이(%p) Top 10')
            display(diff.head(10))

    print(f'\n[{name}] EDA 완료. 결과 저장 위치: {out_dir}\n')
    return {
        'shape': df.shape,
        'missing_summary': miss,
        'binary_cols': binary_cols,
        'categorical_cols': categorical_cols,
        'continuous_cols': continuous_cols,
    }

## 5. 그룹별 EDA 실행

In [ ]:
eda_results = {}
for name, df in dfs.items():
    eda_results[name] = run_eda(name, df)

## 6. 그룹 간 요약 비교

In [ ]:
summary_rows = []
for name, res in eda_results.items():
    summary_rows.append({
        'group': name,
        'n_rows': res['shape'][0],
        'n_cols': res['shape'][1],
        'n_continuous': len(res['continuous_cols']),
        'n_categorical': len(res['categorical_cols']),
        'n_binary_flag': len(res['binary_cols']),
        'n_cols_with_missing': len(res['missing_summary']),
    })

summary_df = pd.DataFrame(summary_rows).set_index('group')
display(summary_df)
summary_df.to_csv(os.path.join(OUTPUT_DIR, 'group_summary.csv'))

## 7. 참고

- 이 노트북은 4개 그룹을 **한 번도 wide 테이블로 병합하지 않습니다**. 3단계의 키 정합성 체크만 `SK_ID_CURR`, `TARGET` 두 컬럼만으로 수행됩니다.
- 이후 모델링 단계에서 그룹을 조인해야 한다면, EDA 노트북이 아닌 별도의 학습 파이프라인에서 `SK_ID_CURR` 기준으로 join하는 것을 권장합니다.
- 결과물(그래프 PNG, 요약 CSV)은 `/content/eda_output/<그룹명>/` 아래에 저장됩니다.